## Tests — Notifications (thresholds + tips + channels)

Validates `model/notify_core.py`:
- 70% / 85% / 95% budget threshold alerts
- overspend / short-runway warnings
- personalized tip from top recommendation
- multi-channel payload (`app`, `email`) and `events_{id}.jsonl` write


## Imports

In [1]:
from __future__ import annotations

import sys
import tempfile
from datetime import date
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").is_dir() and (candidate / "artifacts").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate project root containing data/ and artifacts/")


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from model.notify_core import (
    crossed_thresholds,
    generate_notifications,
    load_events,
    run_notifications_for_client,
    write_events,
)


## Threshold crossing

In [2]:
assert crossed_thresholds(None) == []
assert crossed_thresholds(0.50) == []
assert crossed_thresholds(0.70) == [0.70]
assert crossed_thresholds(0.90) == [0.70, 0.85]
assert crossed_thresholds(0.99) == [0.70, 0.85, 0.95]
print("Threshold crossing: PASS")


Threshold crossing: PASS


## Event generation

In [3]:
as_of = date(2018, 2, 15)

low = generate_notifications(
    client_id=1696,
    as_of_date=as_of,
    utilization_pct=0.28,
    projected_utilization_pct=0.30,
    mtd_spend_usd=600.0,
    monthly_limit_usd=2200.0,
    remaining_budget_usd=1600.0,
    overspend_risk=False,
)
assert low == []

mid = generate_notifications(
    client_id=1696,
    as_of_date=as_of,
    utilization_pct=0.72,
    projected_utilization_pct=0.80,
    mtd_spend_usd=1600.0,
    monthly_limit_usd=2200.0,
    remaining_budget_usd=600.0,
    top_recommendation_title="Pause a subscription",
    top_recommendation_action="Cancel MERCH_SUB (~$13/mo)",
)
kinds = [e.kind for e in mid]
assert kinds.count("budget_threshold") == 1
assert mid[0].threshold_pct == 0.70
assert "tip" in kinds
assert all("app" in e.channels and "email" in e.channels for e in mid)

high = generate_notifications(
    client_id=1696,
    as_of_date=as_of,
    utilization_pct=0.96,
    projected_utilization_pct=1.10,
    mtd_spend_usd=2100.0,
    monthly_limit_usd=2200.0,
    remaining_budget_usd=100.0,
    overspend_risk=True,
)
thr = [e.threshold_pct for e in high if e.kind == "budget_threshold"]
assert thr == [0.95, 0.85, 0.70]
assert any(e.kind == "warning" and e.severity == "critical" for e in high)
assert high[0].severity == "critical"

short = generate_notifications(
    client_id=1696,
    as_of_date=as_of,
    utilization_pct=0.55,
    projected_utilization_pct=0.90,
    days_to_limit_estimate=3,
    remaining_budget_usd=400.0,
    overspend_risk=False,
)
assert any(e.kind == "warning" and "runway" in e.title.lower() for e in short)

print("Event generation: PASS")


Event generation: PASS


## Artifact write / load

In [4]:
with tempfile.TemporaryDirectory() as d:
    root = Path(d)
    (root / "artifacts").mkdir()
    events = generate_notifications(
        client_id=1696,
        as_of_date=date(2018, 2, 15),
        utilization_pct=0.86,
        mtd_spend_usd=1900.0,
        monthly_limit_usd=2200.0,
        remaining_budget_usd=300.0,
    )
    path = write_events(root, client_id=1696, events=events)
    assert path.exists()
    loaded = load_events(root, client_id=1696)
    assert len(loaded) == len(events)
    assert loaded[0]["kind"] == "budget_threshold"

    result = run_notifications_for_client(
        1696,
        root=root,
        as_of_date=date(2018, 2, 15),
        budget={
            "utilization_pct": 0.71,
            "mtd_discretionary_spend_usd": 1560.0,
            "monthly_discretionary_limit_usd": 2200.0,
            "as_of_date": "2018-02-15",
        },
        prediction={"remaining_budget_usd": 640.0, "overspend_risk": False},
        recommendations=[{"title": "Cut dining", "action": "Skip 2 restaurant trips"}],
        write_artifact=True,
    )
    assert result["events_path"]
    assert len(result["app_events"]) >= 1
    assert len(result["email_events"]) >= 1

print("Artifact write/load: PASS")
print("Notify tests: PASS")


Artifact write/load: PASS
Notify tests: PASS
